In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# Vocabulary size
vocab_size = 10000

# Load dataset
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Testing samples: 25000


In [ ]:
# Maximum review length
max_len = 200

# Padding sequences
X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

print("Shape of training data:", X_train.shape)
print("Shape of testing data:", X_test.shape)

Shape of training data: (25000, 200)
Shape of testing data: (25000, 200)


In [ ]:
model = Sequential()

# Embedding layer
model.add(Embedding(input_dim=vocab_size,
                    output_dim=128,
                    input_length=max_len))

# LSTM layer
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))

# Dense layer
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))

# Output layer
model.add(Dense(1, activation='sigmoid'))

# Compile model
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# Model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 199s 617ms/step - accuracy: 0.7370 - loss: 0.5180 - val_accuracy: 0.8246 - val_loss: 0.3908
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 193s 616ms/step - accuracy: 0.8663 - loss: 0.3286 - val_accuracy: 0.8356 - val_loss: 0.3797
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 201s 614ms/step - accuracy: 0.8873 - loss: 0.2886 - val_accuracy: 0.8452 - val_loss: 0.3700
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 201s 609ms/step - accuracy: 0.9115 - loss: 0.2309 - val_accuracy: 0.8506 - val_loss: 0.3891
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 190s 608ms/step - accuracy: 0.9301 - loss: 0.1849 - val_accuracy: 0.8354 - val_loss: 0.4067


In [6]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 58s 74ms/step - accuracy: 0.8423 - loss: 0.3812
Test Loss: 0.3812127411365509
Test Accuracy: 0.8423200249671936


In [7]:
word_index = imdb.get_word_index()

reverse_word_index = {
    value: key for (key, value) in word_index.items()
}

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [8]:
def encode_review(text):
    words = text.lower().split()

    encoded = []

    for word in words:
        index = word_index.get(word)

        if index is not None and index < vocab_size:
            encoded.append(index + 3)

    return pad_sequences([encoded], maxlen=max_len)


def predict_sentiment(review):
    encoded_review = encode_review(review)

    prediction = model.predict(encoded_review)[0][0]

    if prediction >= 0.5:
        print("Positive Review 😊")
    else:
        print("Negative Review 😠")

    print("Confidence Score:", prediction)

In [9]:
predict_sentiment("This movie was absolutely fantastic and amazing")

predict_sentiment("Worst movie ever. Completely boring and waste of time")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 528ms/step
Positive Review 😊
Confidence Score: 0.68549424
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Negative Review 😠
Confidence Score: 0.00835203
